In [ ]:
import torch
from torch_geometric.loader import DataLoader
from dataclasses import dataclass
from pathlib import Path

In [ ]:


@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    base_model_path: Path
    training_data_path: Path
    test_data_path: Path
    params_epochs: int
    params_batch_size: int
    params_learning_rate: float



In [ ]:
from src.GNNClassfier.constants import *
from src.GNNClassfier.utils.common import read_yaml, create_directories

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        
        # read_yaml returns a ConfigBox for attribute-style access
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        
        create_directories([self.config.artifacts_root])
        
    def get_training_config(self) -> TrainingConfig:
        training_config = TrainingConfig(
            root_dir = self.config.artifacts_root,
            trained_model_path = self.config.trained_model_path,
            base_model_path = self.config.base_model_path,
            training_data_path = self.config.training_data_path,
            test_data_path = self.config.test_data_path,
            params_epochs = self.params.EPOCHS,
            params_batch_size = self.params.BATCH_SIZE,
            params_learning_rate = self.params.LEARNING_RATE
        )
        return training_config

In [ ]:
class ModelTrainer:
    def __init__(self, config: TrainingConfig):
        self.config = config
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def train_model(self):
        # 1. Load Data
        train_data = torch.load(self.config.training_data_path)
        test_data = torch.load(self.config.test_data_path)
        
        train_loader = DataLoader(train_data, batch_size=self.config.params_batch_size, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=self.config.params_batch_size, shuffle=False)

        # 2. Load Model
        model = torch.load(self.config.base_model_path).to(self.device)
        optimizer = torch.optim.Adam(model.parameters(), lr=self.config.params_learning_rate)
        criterion = torch.nn.CrossEntropyLoss()

        # 3. Training Loop
        print(f"Starting training on {self.device} for {self.config.params_epochs} epochs...")
        for epoch in range(1, self.config.params_epochs + 1):
            model.train()
            total_loss = 0
            for data in train_loader:
                data = data.to(self.device)
                optimizer.zero_grad()
                out = model(data.x.float(), data.edge_index, data.batch)
                loss = criterion(out, data.y.long().squeeze())
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            
            # Validation
            model.eval()
            correct = 0
            with torch.no_grad():
                for data in test_loader:
                    data = data.to(self.device)
                    out = model(data.x.float(), data.edge_index, data.batch)
                    pred = out.argmax(dim=1)
                    correct += (pred == data.y.long().squeeze()).sum().item()
            
            acc = correct / len(test_data)
            print(f"Epoch {epoch:03d} | Loss: {total_loss/len(train_loader):.4f} | Test Acc: {acc:.4f}")

        # 4. Save Trained Model
        torch.save(model.state_dict(), self.config.trained_model_path)
        print(f"Trained weights saved to: {self.config.trained_model_path}")

In [ ]:
try:
    config_manager = ConfigurationManager()
    training_config = config_manager.get_training_config()
    trainer = ModelTrainer(training_config)
    trainer.train_model()
except Exception as e:
    print(f"Error during training: {e}")
    